## Импорты и конфигурация

In [1]:
import argparse
import gc
import os
import warnings
from dataclasses import dataclass
from typing import Dict, List, Optional, Tuple

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import torch

from neuralforecast import NeuralForecast
from neuralforecast.models import NHITS
from sklearn.metrics import r2_score

warnings.filterwarnings("ignore")


@dataclass
class ImputeConfig:
    step_seconds: int = 3
    q_threshold: int = 2
    storm_rolling_window_hours: int = 24
    storm_std_factor: float = 2.0
    storm_min_periods: int = 1000

    # Пороги каскада (в минутах)
    short_gap_max_minutes: float = 3.0          # < 3 мин => линейная интерполяция
    medium_gap_max_minutes: float = 90.0        # до 90 мин => NHITS (возможно рекурсивно)

    # Длина чанка/горизонта для NHITS (в минутах); обучаем одну модель и переиспользуем
    nhits_chunk_minutes: float = 60.0          # 60 мин = 1200 шагов (для step_seconds=3)
    input_size: int = 256
    max_steps: int = 350
    batch_size: int = 16
    windows_batch_size: int = 512
    seed: int = 42

    # Ограничения на обучающие данные (чтобы не взорвать GPU/VRAM)
    max_series_per_label: int = 60
    max_points_per_series: int = 12000

    # Стратегия обработки больших пропусков
    long_gap_mode: str = "A_recursive"  # A_recursive | C_overlap
    long_overlap_minutes: float = 15.0  # используется только при long_gap_mode=C_overlap

    # Параметры оценки качества (сколько сегментов пробовать на маскировках)
    eval_segments_per_bucket: int = 3
    eval_random_seed: int = 123

    # Фолбэк (если модель не смогла предсказать)
    fallback_hourly_median: bool = True


def parse_args() -> argparse.Namespace:
    return argparse.Namespace()
RUN_YEAR = 2024
CSV_PATH = "ARS_pos1_2024.csv"
OUT_DIR = "out_nhits_2024"
DEVICE_CLEAR_CUDA = True

## Вспомогательные функции: загрузка данных

In [2]:
def build_seconds_grid(seconds: np.ndarray, step_seconds: int) -> Tuple[np.ndarray, np.ndarray]:
    """
    Собирает регулярную сетку времени с шагом `step_seconds`.

    Возвращает:
      - `seconds_full`: вся регулярная сетка (от min до max)
      - `idx_map`: отображение индекса исходной точки в индекс регулярной сетки
    """
    seconds = seconds.astype(np.int64)
    s0, s1 = int(seconds.min()), int(seconds.max())
    n = (s1 - s0) // step_seconds + 1
    seconds_full = s0 + np.arange(n, dtype=np.int64) * step_seconds
    idx_map = (seconds - s0) // step_seconds
    return seconds_full, idx_map


def load_ars_series(csv_path: str, year: int, cfg: ImputeConfig) -> Tuple[np.ndarray, np.ndarray, np.ndarray, np.ndarray]:
    """
    Возвращает:
      seconds_full: int64 массив регулярной сетки
      ds_full: datetime64 массив времени для этой сетки
      y_raw: массив значений (NaN там, где значение не наблюдалось/невалидно)
      is_bad_mask: маска, что считать пропуском (плохое качество/битые точки)
    """
    df = pd.read_csv(
        csv_path,
        header=None,
        names=["seconds", "value", "quality", "flag"],
        on_bad_lines="skip",
        engine="python",
    )
    df = df[df["quality"] >= 0].reset_index(drop=True)
    df = df[df["seconds"].diff().fillna(1) > 0].reset_index(drop=True)

    # Строим полную регулярную сетку по `step_seconds`
    seconds_full, idx_map = build_seconds_grid(df["seconds"].values, cfg.step_seconds)
    n_full = len(seconds_full)

    y_raw = np.full(n_full, np.nan, dtype=np.float32)
    quality_arr = np.full(n_full, np.nan, dtype=np.float32)
    flag_arr = np.zeros(n_full, dtype=np.int64)

    # Заполняем массивы значениями (если дубликаты — последнее значение будет использовано)
    seconds0 = int(seconds_full[0])
    idx = idx_map.astype(np.int64)
    valid_idx = (idx >= 0) & (idx < n_full)
    idx = idx[valid_idx]
    vals = df.loc[valid_idx, "value"].values.astype(np.float32)
    y_raw[idx] = vals
    quality_arr[idx] = df.loc[valid_idx, "quality"].values.astype(np.float32)
    flag_arr[idx] = df.loc[valid_idx, "flag"].values.astype(np.int64)

    # Считаем точку "плохой/пропуском", если:
    #  - quality > порога
    #  - flag != 0 (признак проблемы)
    #  - или значение не попало в регулярную сетку (y_raw == NaN)
    is_bad = (flag_arr != 0) | (quality_arr > cfg.q_threshold) | np.isnan(y_raw)

    base = pd.Timestamp(f"{year}-01-01")
    ds_full = base + pd.to_timedelta(seconds_full, unit="s")
    return seconds_full, ds_full.to_numpy(), y_raw, is_bad


def compute_is_storm(y_clean: np.ndarray, cfg: ImputeConfig, step_seconds: int) -> np.ndarray:
    """
    Replicates logic from fill_gap_csv_2024.ipynb:
      rolling_std_24h = value_clean.rolling(WINDOW_24H, min_periods=1000).std()
      threshold = median(rolling_std_24h) * 2
      is_storm = rolling_std_24h > threshold
    """
    window = (cfg.storm_rolling_window_hours * 3600) // step_seconds
    s = pd.Series(y_clean)
    rolling_std = s.rolling(window=window, min_periods=cfg.storm_min_periods).std()
    thr = float(rolling_std.median(skipna=True) * cfg.storm_std_factor)
    is_storm = (rolling_std > thr).fillna(False).to_numpy()
    return is_storm

## Подготовка сегментов для обучения

In [3]:
def split_into_segments(
    y_clean: np.ndarray,
    is_storm: np.ndarray,
    ds: np.ndarray,
    input_size: int,
    h_train: int,
    max_series: int,
    max_points_per_series: int,
) -> Dict[str, pd.DataFrame]:
    """
    Готовит данные для neuralforecast отдельно для спокойных (calm) и штормовых (storm) периодов:
    - собираем непрерывные участки, где `y_clean` не NaN
    - внутри участка дополнительно делим по переключению режима storm/calm
    """
    valid = ~np.isnan(y_clean)
    # Ищем непрерывные участки наблюдений без NaN
    starts = np.where((valid[1:] & ~valid[:-1]))[0] + 1
    ends = np.where((~valid[1:] & valid[:-1]))[0] + 1
    if valid[0]:
        starts = np.r_[0, starts]
    if valid[-1]:
        ends = np.r_[ends, len(valid)]

    segments: List[Tuple[str, int, int]] = []
    for s, e in zip(starts, ends):
        if e - s < input_size + h_train:
            continue
        # Делим по переключению метки storm/calm внутри непрерывного участка
        labels = is_storm[s:e]
        d = np.diff(labels.astype(np.int8))
        split_points = np.where(d != 0)[0] + 1
        sub_starts = np.r_[s, s + split_points]
        sub_ends = np.r_[s + split_points, e]
        for ss, ee in zip(sub_starts, sub_ends):
            if ee - ss < input_size + h_train:
                continue
            label = "storm" if is_storm[ss] else "calm"
            segments.append((label, ss, ee))

    # Ограничиваем число сегментов на каждый класс (calm/storm)
    out: Dict[str, List[pd.DataFrame]] = {"calm": [], "storm": []}
    counts = {"calm": 0, "storm": 0}
    for label, ss, ee in segments:
        if counts[label] >= max_series:
            continue
        # Оставляем "хвост" сегмента, чтобы снизить число точек и сохранить недавнюю динамику
        if ee - ss > max_points_per_series:
            ee = ee
            ss = ee - max_points_per_series
        if ee - ss < input_size + h_train:
            continue

        uid = f"{label}_{counts[label]}"
        df_seg = pd.DataFrame({"unique_id": uid, "ds": ds[ss:ee], "y": y_clean[ss:ee].astype(np.float32)})
        out[label].append(df_seg)
        counts[label] += 1

    calm_df = pd.concat(out["calm"], ignore_index=True) if out["calm"] else pd.DataFrame(columns=["unique_id", "ds", "y"])
    storm_df = pd.concat(out["storm"], ignore_index=True) if out["storm"] else pd.DataFrame(columns=["unique_id", "ds", "y"])
    return {"calm": calm_df, "storm": storm_df}

## Обучение NHITS

In [4]:
def train_nhits(nf_df: pd.DataFrame, h_train: int, cfg: ImputeConfig) -> NeuralForecast:
    # HuberLoss обычно стабильнее MAE при шуме/выбросах.
    from neuralforecast.losses.pytorch import HuberLoss

    model = NHITS(
        h=h_train,
        input_size=cfg.input_size,
        loss=HuberLoss(),
        max_steps=cfg.max_steps,
        batch_size=cfg.batch_size,
        windows_batch_size=cfg.windows_batch_size,
        random_seed=cfg.seed,
    )
    nf = NeuralForecast(models=[model], freq=f"{cfg.step_seconds}s")
    nf.fit(df=nf_df)
    return nf

## Вспомогательные функции: заполнение пропусков

In [5]:
def compute_hourly_medians(ds: np.ndarray, y_raw: np.ndarray, is_bad: np.ndarray) -> Dict[Tuple[int, int], float]:
    good = ~is_bad & ~np.isnan(y_raw)
    dsg = pd.to_datetime(ds[good])
    x = y_raw[good]
    df = pd.DataFrame({"month": dsg.month, "hour": dsg.hour, "y": x})
    med = df.groupby(["month", "hour"])["y"].median()
    return {(int(m), int(h)): float(v) for (m, h), v in med.items()}


def find_nan_runs(mask: np.ndarray) -> List[Tuple[int, int]]:
    mask = np.asarray(mask, dtype=bool)
    padded = np.concatenate([[False], mask, [False]])
    diff   = np.diff(padded.astype(np.int8))
    starts = np.where(diff ==  1)[0]
    ends   = np.where(diff == -1)[0]
    return list(zip(starts.tolist(), ends.tolist()))


def linear_fill_gap(
    y: np.ndarray,
    ds: np.ndarray,
    start: int,
    end: int,
) -> Tuple[np.ndarray, np.ndarray]:
    """
    Заполняет участок `y[start:end]` (в нем все NaN) линейной интерполяцией между
    `y[start-1]` и `y[end]`.

    Возвращает:
      - заполненные значения
      - `method_codes`: код метода (1 для интерполяции)
    """
    gap_len = end - start
    left_idx = start - 1
    right_idx = end
    if left_idx < 0 or right_idx >= len(y) or np.isnan(y[left_idx]) or np.isnan(y[right_idx]):
        # Если интерполяция невозможна (нет опорных точек) — заполняем константой ближайшей стороны
        if left_idx >= 0 and not np.isnan(y[left_idx]):
            vals = np.full(gap_len, y[left_idx], dtype=np.float32)
        elif right_idx < len(y) and not np.isnan(y[right_idx]):
            vals = np.full(gap_len, y[right_idx], dtype=np.float32)
        else:
            vals = np.zeros(gap_len, dtype=np.float32)
        codes = np.ones(gap_len, dtype=np.uint8)
        return vals, codes

    left_val = float(y[left_idx])
    right_val = float(y[right_idx])
    # Коэффициент (k+1)/(gap_len+1) дает стандартную линейную интерполяцию внутри разрыва
    t = (np.arange(gap_len, dtype=np.float32) + 1.0) / (gap_len + 1.0)
    vals = (1.0 - t) * left_val + t * right_val
    codes = np.ones(gap_len, dtype=np.uint8)
    return vals.astype(np.float32), codes


def nhits_predict_chunk(nf: NeuralForecast, input_ctx: np.ndarray, ds_ctx: np.ndarray) -> np.ndarray:
    nf_input = pd.DataFrame({"unique_id": "mag", "ds": ds_ctx, "y": input_ctx})
    pred = nf.predict(df=nf_input)
    pred_col = [c for c in pred.columns if c not in ["unique_id", "ds"]][0]
    return pred[pred_col].to_numpy(dtype=np.float32)


def impute_gap(
    y_inout: np.ndarray,
    ds: np.ndarray,
    is_storm: np.ndarray,
    nf_calm: NeuralForecast,
    nf_storm: NeuralForecast,
    hourly_medians: Dict[Tuple[int, int], float],
    gap_start: int,
    gap_end: int,
    cfg: ImputeConfig,
    method_codes: np.ndarray,
) -> None:
    """
    Заполняет (in-place) `y_inout[gap_start:gap_end]` (там все NaN).

    `method_codes` заполняется для всех индексов внутри пропуска:
      1 = короткий разрыв: линейная интерполяция
      2 = разрыв: предсказание NHITS
      3 = фолбэк: медиана по часу суток
    """
    gap_len = gap_end - gap_start
    step_seconds = cfg.step_seconds
    short_gap_steps = int((cfg.short_gap_max_minutes * 60) // step_seconds)
    medium_gap_steps = int((cfg.medium_gap_max_minutes * 60) // step_seconds)
    h_train_steps = int((cfg.nhits_chunk_minutes * 60) // step_seconds)
    overlap_steps = int((cfg.long_overlap_minutes * 60) // step_seconds)
    if overlap_steps >= h_train_steps:
        overlap_steps = max(0, h_train_steps // 4)

    # 1) Короткие пропуски => линейная интерполяция
    if gap_len <= short_gap_steps:
        vals, codes = linear_fill_gap(y_inout, ds, gap_start, gap_end)
        y_inout[gap_start:gap_end] = vals
        method_codes[gap_start:gap_end] = codes
        return

    # Вспомогательная функция: заполнение медианой по часу суток
    def fallback_hourly():
        for i in range(gap_start, gap_end):
            di = pd.to_datetime(ds[i])
            key = (int(di.month), int(di.hour))
            v = hourly_medians.get(key, None)
            if v is None:
                v = 0.0
            y_inout[i] = v
            method_codes[i] = 3

    # Проверяем, что контекст для NHITS не содержит NaN и достаточно длинный
    def get_context(ctx_end: int) -> Optional[Tuple[np.ndarray, np.ndarray]]:
        ctx_start = ctx_end - cfg.input_size
        if ctx_start < 0:
            return None
        y_ctx = y_inout[ctx_start:ctx_end]
        if np.isnan(y_ctx).any():
            return None
        ds_ctx = ds[ctx_start:ctx_end]
        return y_ctx.astype(np.float32, copy=False), ds_ctx

    # 2) Средние (<=90мин) и 3) Длинные (>90мин): заполняем через NHITS (каскадно)
    if gap_len <= medium_gap_steps:
        # Рекурсивное прогнозирование, если разрыв не помещается в один чанк
        cur = gap_start
        while cur < gap_end:
            ctx = get_context(cur)
            if ctx is None:
                fallback_hourly()
                return
            # Выбираем модель по метке storm/calm на последней точке контекста
            mode_is_storm = bool(is_storm[cur - 1]) if cur - 1 >= 0 else False
            nf = nf_storm if mode_is_storm else nf_calm
            y_ctx, ds_ctx = ctx
            pred_vals = nhits_predict_chunk(nf, y_ctx, ds_ctx)
            chunk_len = min(h_train_steps, gap_end - cur)
            y_inout[cur:cur + chunk_len] = pred_vals[:chunk_len]
            method_codes[cur:cur + chunk_len] = 2
            cur += chunk_len
        return

    # Разрыв длиннее medium threshold (обработка стратегией long_gap_mode)
    if cfg.long_gap_mode == "A_recursive":
        cur = gap_start
        while cur < gap_end:
            ctx = get_context(cur)
            if ctx is None:
                fallback_hourly()
                return
            mode_is_storm = bool(is_storm[cur - 1]) if cur - 1 >= 0 else False
            nf = nf_storm if mode_is_storm else nf_calm
            y_ctx, ds_ctx = ctx
            pred_vals = nhits_predict_chunk(nf, y_ctx, ds_ctx)
            chunk_len = min(h_train_steps, gap_end - cur)
            y_inout[cur:cur + chunk_len] = pred_vals[:chunk_len]
            method_codes[cur:cur + chunk_len] = 2
            cur += chunk_len
        return

    # C_overlap: перекрывающиеся сегменты длиной `h_train_steps`
    # Предсказания в перекрытиях усредняем, чтобы снизить накопление ошибок
    if cfg.long_gap_mode == "C_overlap":
        stride = max(1, h_train_steps - overlap_steps)
        acc_sum = np.zeros(gap_len, dtype=np.float64)
        acc_cnt = np.zeros(gap_len, dtype=np.float64)

        for seg_start in range(gap_start, gap_end, stride):
            ctx = get_context(seg_start)
            if ctx is None:
                continue
            mode_is_storm = bool(is_storm[seg_start - 1]) if seg_start - 1 >= 0 else False
            nf = nf_storm if mode_is_storm else nf_calm
            y_ctx, ds_ctx = ctx
            pred_vals = nhits_predict_chunk(nf, y_ctx, ds_ctx)
            seg_end = min(seg_start + h_train_steps, gap_end)
            pred_len = seg_end - seg_start
            tgt_l = seg_start - gap_start
            # Временно пишем прогноз в y_inout, чтобы следующие сегменты могли построить контекст
            y_inout[seg_start:seg_end] = pred_vals[:pred_len]

            acc_sum[tgt_l:tgt_l + pred_len] += pred_vals[:pred_len].astype(np.float64, copy=False)
            acc_cnt[tgt_l:tgt_l + pred_len] += 1.0

        ok = acc_cnt > 0
        if not ok.any():
            fallback_hourly()
            return

        rel_idx = np.where(ok)[0].astype(np.int64)
        y_inout[gap_start + rel_idx] = (acc_sum[ok] / acc_cnt[ok]).astype(np.float32)
        method_codes[gap_start + rel_idx] = 2

        # Точки без предсказаний => фолбэк медианой по часу суток
        if (~ok).any():
            for rel_i in np.where(~ok)[0]:
                i = gap_start + int(rel_i)
                di = pd.to_datetime(ds[i])
                key = (int(di.month), int(di.hour))
                v = hourly_medians.get(key, None)
                if v is None:
                    v = 0.0
                y_inout[i] = v
                method_codes[i] = 3
        return

    # Unknown mode
    fallback_hourly()


def r2_score_safe(y_true: np.ndarray, y_pred: np.ndarray) -> float:
    if len(y_true) < 2:
        return np.nan
    # Если дисперсия нулевая, R2 становится некорректным => возвращаем NaN
    if np.allclose(np.var(y_true), 0.0):
        return np.nan
    return float(r2_score(y_true, y_pred))

## Оценка качества

In [6]:
def eval_imputation(
    y_clean: np.ndarray,
    ds: np.ndarray,
    is_storm: np.ndarray,
    nf_calm: NeuralForecast,
    nf_storm: NeuralForecast,
    hourly_medians: Dict[Tuple[int, int], float],
    cfg: ImputeConfig,
) -> pd.DataFrame:
    """
    Оценка качества имputation через синтетическое маскирование:
    - выбираем непрерывные сегменты валидных точек (где y_clean не NaN)
    - маскируем только этот сегмент (как будто там пропуск)
    - заполняем пропуск и считаем ошибки MAE/RMSE/R2
    """
    # Бакеты по длительности пропуска (в минутах)
    buckets = [
        ("<15m", 0.0, 15.0),
        ("15-60m", 15.0, 60.0),
        ("1-3h", 60.0, 180.0),
        (">3h", 180.0, 1e9),
    ]
    rng = np.random.default_rng(cfg.eval_random_seed)

    good_mask = ~np.isnan(y_clean)
    N = len(y_clean)

    results = []

    # Для каждого бакета подбираем несколько стартовых позиций (простая стохастика)
    for bucket_name, lo_min, hi_min in buckets:
        L_min = int(np.ceil((lo_min * 60) / cfg.step_seconds))
        L_max = int(np.floor((hi_min * 60) / cfg.step_seconds))
        if L_max <= L_min:
            continue

        # Choose a few representative lengths inside bucket
        lengths = sorted(
            set(
                [
                    max(L_min + 1, int(L_min * 1.1)),
                    max(L_min + 1, int((L_min + L_max) / 2)),
                    max(L_min + 1, min(L_max, int(L_max * 0.9))),
                ]
            )
        )

        maes = []
        r2s = []
        cover_model_cnt = 0
        cover_model_total = 0

        for L in lengths:
            # пробуем случайные стартовые позиции
            tries = 0
            picked = 0
            while picked < cfg.eval_segments_per_bucket and tries < cfg.eval_segments_per_bucket * 50:
                tries += 1
                high = N - L - 1
                if high <= cfg.input_size + 1:
                    break
                start = int(rng.integers(low=cfg.input_size + 1, high=high))
                end = start + L
                # Пропуск должен целиком лежать внутри наблюдаемых значений
                if not good_mask[start:end].all():
                    continue
                # Контекст (входное окно NHITS) тоже должен быть без NaN
                if not good_mask[start - cfg.input_size : start].all():
                    continue
                # Для коротких пропусков правый край также должен быть наблюдаемым
                if L <= int((cfg.short_gap_max_minutes * 60) // cfg.step_seconds):
                    if not good_mask[end]:
                        continue

                # Create masked series
                y_test = y_clean.copy()
                y_test[start:end] = np.nan
                methods = np.zeros_like(y_test, dtype=np.uint8)
                # Отмечаем позиции пропуска (для подсчета coverage модели)
                methods[start:end] = 0
                gap_mask = np.isnan(y_test)
                assert gap_mask[start:end].all()

                # Импутируем только этот выбранный разрыв
                impute_gap(
                    y_inout=y_test,
                    ds=ds,
                    is_storm=is_storm,
                    nf_calm=nf_calm,
                    nf_storm=nf_storm,
                    hourly_medians=hourly_medians,
                    gap_start=start,
                    gap_end=end,
                    cfg=cfg,
                    method_codes=methods,
                )

                y_pred = y_test[start:end]
                y_true = y_clean[start:end]
                mae = float(np.mean(np.abs(y_true - y_pred)))
                rmse = float(np.sqrt(np.mean((y_true - y_pred) ** 2)))
                r2 = r2_score_safe(y_true, y_pred)
                maes.append((L, mae, rmse))
                r2s.append((L, r2))

                # Подсчет coverage модели NHITS среди пропущенных точек
                model_used = (methods[start:end] == 2).sum()
                # coverage считаем только там, где по стратегии должны были применять модель
                short_gap_steps = int((cfg.short_gap_max_minutes * 60) // cfg.step_seconds)
                if L > short_gap_steps:
                    cover_model_cnt += int(model_used)
                    cover_model_total += int(L)
                picked += 1

            if picked == 0:
                continue

        if cover_model_total == 0:
            cover = np.nan
        else:
            cover = cover_model_cnt / cover_model_total

        if len(maes) == 0:
            continue

        mae_vals = [m[1] for m in maes]
        rmse_vals = [m[2] for m in maes]
        r2_vals = [r[1] for r in r2s if not np.isnan(r[1])]

        results.append(
            {
                "bucket": bucket_name,
                "gap_minutes_mid": round((lo_min + hi_min) / 2, 2) if hi_min < 1e9 else np.nan,
                "n_samples": len(mae_vals),
                "MAE": float(np.mean(mae_vals)),
                "RMSE": float(np.mean(rmse_vals)),
                "R2": float(np.mean(r2_vals)) if len(r2_vals) else np.nan,
                "coverage_model": float(cover),
            }
        )

    return pd.DataFrame(results)


def plot_gap_example(
    ds: np.ndarray,
    y_obs: np.ndarray,
    y_filled: np.ndarray,
    gap_runs: List[Tuple[int, int]],
    cfg: ImputeConfig,
    target_minutes: float = 81.0,
    out_path: str = "gap_example.png",
) -> None:
    """
    Рисует наблюдаемый ряд (NaN показаны как пропуски) и итоговую имputation
    на примере одного representative-пропуска.
    """
    step_seconds = cfg.step_seconds
    if not gap_runs:
        return

    gap_infos = []
    for s, e in gap_runs:
        L = e - s
        minutes = L * step_seconds / 60.0
        gap_infos.append((abs(minutes - target_minutes), s, e, minutes))
    gap_infos.sort(key=lambda x: x[0])
    _, gap_start, gap_end, minutes = gap_infos[0]

    pad_minutes = 60.0
    pad_steps = int((pad_minutes * 60) // step_seconds)
    w_start = max(0, gap_start - pad_steps)
    w_end = min(len(ds), gap_end + pad_steps)

    x = pd.to_datetime(ds[w_start:w_end])
    y0 = y_obs[w_start:w_end]
    y1 = y_filled[w_start:w_end]

    plt.figure(figsize=(14, 5))
    plt.plot(x, y0, label="Исходные данные", linewidth=1.5, color="#1f77b4")
    plt.plot(x, y1, label="Импутация (NHITS cascade)", linewidth=1.5, color="#d62728", alpha=0.9)

    miss_l = gap_start - w_start
    miss_r = gap_end - w_start
    if miss_l < miss_r:
        plt.axvspan(x[miss_l], x[miss_r - 1], color="red", alpha=0.08, label=f"Пропуск ~{minutes:.0f} мин")
    plt.title(f"Импутация пропуска ~{minutes:.0f} мин ({cfg.long_gap_mode})")
    plt.xlabel("Время")
    plt.ylabel("Значение")
    plt.grid(alpha=0.3)
    plt.legend()
    plt.tight_layout()

    os.makedirs(os.path.dirname(out_path) or ".", exist_ok=True)
    plt.savefig(out_path, dpi=150)
    plt.close()


def plot_hour_slice(
    ds: np.ndarray,
    y_obs: np.ndarray,
    y_filled: np.ndarray,
    gap_runs: List[Tuple[int, int]],
    cfg: ImputeConfig,
    out_path: str = "hour_slice.png",
) -> None:
    """
    Рисует ровно один час внутри выбранного пропуска для визуальной проверки.
    """
    if not gap_runs:
        return
    step_seconds = cfg.step_seconds
    h_steps = int((60 * 60) // step_seconds)

    gap_runs_sorted = sorted(gap_runs, key=lambda x: (x[1] - x[0]), reverse=True)
    gap_start, gap_end = gap_runs_sorted[0]
    L = gap_end - gap_start
    if L < h_steps:
        return

    slice_start = gap_start + (L // 2) - (h_steps // 2)
    slice_end = slice_start + h_steps
    pad_steps = int((30 * 60) // step_seconds)
    w_start = max(0, slice_start - pad_steps)
    w_end = min(len(ds), slice_end + pad_steps)

    x = pd.to_datetime(ds[w_start:w_end])
    y0 = y_obs[w_start:w_end]
    y1 = y_filled[w_start:w_end]

    plt.figure(figsize=(14, 5))
    plt.plot(x, y0, label="Исходные данные", linewidth=1.5, color="#1f77b4")
    plt.plot(x, y1, label="Импутация", linewidth=1.5, color="#d62728", alpha=0.9)

    miss_l = slice_start - w_start
    miss_r = slice_end - w_start
    plt.axvspan(x[miss_l], x[miss_r - 1], color="red", alpha=0.08, label="Окно 1 час")

    plt.title("Заполнение одного часа (визуальная проверка)")
    plt.xlabel("Время")
    plt.ylabel("Значение")
    plt.grid(alpha=0.3)
    plt.legend()
    plt.tight_layout()

    os.makedirs(os.path.dirname(out_path) or ".", exist_ok=True)
    plt.savefig(out_path, dpi=150)
    plt.close()


## Визуализация

In [7]:
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
import matplotlib.dates as mdates
import matplotlib.gridspec as gridspec
import matplotlib.ticker as mticker
import pandas as pd
import numpy as np
import os

OUT_DIR = "out_nhits_2024"
os.makedirs(OUT_DIR, exist_ok=True)

C = {
    "obs":    "#2c7bb6",
    "nhits":  "#d7191c",
    "interp": "#fdae61",
    "median": "#1a9641",
    "ffill":  "#7b2d8b",
    "gap_bg": "#fee8e8",
}

plt.rcParams.update({
    "font.family":       "DejaVu Sans",
    "axes.grid":         True,
    "grid.alpha":        0.3,
    "grid.linestyle":    "--",
    "axes.spines.top":   False,
    "axes.spines.right": False,
    "figure.facecolor":  "white",
    "axes.facecolor":    "#fafafa",
})


def _fmt_xaxis(ax, x_arr):
    span_h = (x_arr[-1] - x_arr[0]).total_seconds() / 3600
    if span_h < 3:
        fmt = "%H:%M"
    elif span_h < 72:
        fmt = "%m-%d %H:%M"
    else:
        fmt = "%Y-%m-%d"
    ax.xaxis.set_major_formatter(mdates.DateFormatter(fmt))
    plt.setp(ax.xaxis.get_majorticklabels(), rotation=15, ha="right")


def _get_runs(method_codes):
    mc = np.asarray(method_codes, dtype=bool)
    padded = np.concatenate([[False], mc, [False]])
    diff   = np.diff(padded.astype(np.int8))
    starts = np.where(diff ==  1)[0].tolist()
    ends   = np.where(diff == -1)[0].tolist()
    return list(zip(starts, ends))


# ── График 1: сравнение методов на одном пропуске ──────────────────────────
def fig_comparison(ds, y_clean, y_filled, method_codes,
                   nf_calm, nf_storm, hourly_medians, is_storm,
                   cfg, gap_hours=1.0, out_dir=OUT_DIR):
    step    = cfg.step_seconds
    runs    = _get_runs(method_codes > 0)
    if not runs:
        print(f"[fig_comparison] No filled runs for gap_hours={gap_hours}")
        return

    target_steps = int(gap_hours * 3600 / step)
    best_run = min(runs, key=lambda r: abs((r[1] - r[0]) - target_steps))
    gap_s, gap_e = best_run
    actual_min = (gap_e - gap_s) * step / 60

    ctx_len  = cfg.input_size
    ctx_s    = max(0, gap_s - ctx_len)
    y_ctx    = y_clean[ctx_s:gap_s].astype(float)
    ds_ctx   = ds[ctx_s:gap_s]
    gt       = y_clean[gap_s:gap_e].astype(float)

    # Метод 1: ffill
    filled_ff = np.full(gap_e - gap_s, np.nan, dtype=float)
    if gap_s > 0 and not np.isnan(y_clean[gap_s - 1]):
        filled_ff[:] = float(y_clean[gap_s - 1])

    # Метод 2: линейная интерполяция
    filled_li = np.full(gap_e - gap_s, np.nan, dtype=float)
    if gap_s > 0 and gap_e < len(y_clean):
        lv = float(y_clean[gap_s - 1]) if not np.isnan(y_clean[gap_s - 1]) else np.nan
        rv = float(y_clean[gap_e])     if not np.isnan(y_clean[gap_e])     else np.nan
        if not np.isnan(lv) and not np.isnan(rv):
            t = np.linspace(0, 1, gap_e - gap_s + 2)[1:-1]
            filled_li = lv + t * (rv - lv)

    # Метод 3: NHITS (берём из y_filled)
    filled_nhits = y_filled[gap_s:gap_e].astype(float)

    def mae_rmse(pred):
        ok = ~np.isnan(pred) & ~np.isnan(gt)
        if ok.sum() < 2:
            return np.nan, np.nan
        return (np.nanmean(np.abs(pred[ok] - gt[ok])),
                np.sqrt(np.nanmean((pred[ok] - gt[ok])**2)))

    methods = {
        "ffill":         (filled_ff,    C["ffill"],   "--"),
        "Интерполяция":  (filled_li,    C["interp"],  "-"),
        "NHITS":         (filled_nhits, C["nhits"],   "-"),
    }
    metrics = {name: mae_rmse(arr) for name, (arr, _, _) in methods.items()}
    best_name = min(metrics, key=lambda k: metrics[k][0] if not np.isnan(metrics[k][0]) else 1e18)

    pad   = int(min(240, max(60, actual_min * 2)) * 60 / step)
    w0    = max(0, gap_s - pad)
    w1    = min(len(ds), gap_e + pad)
    x_all = pd.to_datetime(ds[w0:w1])
    y_all = y_clean[w0:w1].astype(float)

    fig, (ax, ax_tbl) = plt.subplots(
        2, 1, figsize=(14, 9),
        gridspec_kw={"height_ratios": [3, 1]}
    )
    fig.suptitle(f"Заполнение пропуска ~{actual_min:.0f} мин",
                 fontsize=13, fontweight="bold")

    ax.plot(x_all, y_all, color=C["obs"], lw=0.9, alpha=0.4,
            label="оригинал", zorder=2)

    xl = pd.to_datetime(ds[gap_s])
    xr = pd.to_datetime(ds[min(gap_e, len(ds)-1)])
    ax.axvspan(xl, xr, color=C["gap_bg"], alpha=0.8, zorder=1, label="пропуск")

    x_gap = pd.to_datetime(ds[gap_s:gap_e])
    for name, (arr, color, ls) in methods.items():
        mae_v, _ = metrics[name]
        lbl = f"{name}  MAE={mae_v:,.0f}" if not np.isnan(mae_v) else name
        ax.plot(x_gap, arr, color=color, lw=2.0, ls=ls, label=lbl, zorder=5)

    ax.set_ylabel("Магнитное поле, нТл", fontsize=11)
    ax.legend(loc="upper left", fontsize=9, framealpha=0.9)
    _fmt_xaxis(ax, x_all)

    ax_tbl.axis("off")
    rows = []
    for name in methods:
        mae_v, rmse_v = metrics[name]
        mark = " ✓" if name == best_name else ""
        rows.append([
            f"{name}{mark}",
            f"{mae_v:,.1f}" if not np.isnan(mae_v) else "—",
            f"{rmse_v:,.1f}" if not np.isnan(rmse_v) else "—",
        ])

    tbl = ax_tbl.table(
        cellText=rows,
        colLabels=["Метод", "MAE, нТл", "RMSE, нТл"],
        cellLoc="center", loc="center",
        bbox=[0.2, 0.0, 0.6, 1.0],
    )
    tbl.auto_set_font_size(False)
    tbl.set_fontsize(11)
    for j in range(3):
        tbl[0, j].set_facecolor("#d0d8e8")
        tbl[0, j].set_text_props(fontweight="bold")
    for i, name in enumerate(methods):
        if name == best_name:
            for j in range(3):
                tbl[i+1, j].set_facecolor("#c8f0c8")

    fig.tight_layout()
    out = os.path.join(out_dir, f"comparison_{actual_min:.0f}min.png")
    fig.savefig(out, dpi=150, bbox_inches="tight")
    plt.close(fig)
    print(f"Saved: {out}")


# ── График 2: три примера заполнения (короткий / средний / длинный) ─────────
def fig_examples(ds, y_clean, y_filled, method_codes, cfg, out_dir=OUT_DIR):
    step = cfg.step_seconds
    runs = _get_runs(method_codes > 0)
    if not runs:
        print("[fig_examples] No runs, skipping.")
        return

    run_info = sorted([(r[1]-r[0], r) for r in runs], key=lambda x: x[0])
    n = len(run_info)

    targets = []
    # короткий
    short = [r for sz, r in run_info if sz * step / 60 < 5]
    if short: targets.append(("Короткий (<5 мин)", short[-1]))
    # средний
    mid = [r for sz, r in run_info if 5 <= sz * step / 60 < 60]
    if mid: targets.append(("Средний (5–60 мин)", mid[len(mid)//2]))
    # длинный
    lng = [r for sz, r in run_info if sz * step / 60 >= 60]
    if lng: targets.append(("Длинный (>60 мин)", lng[-1]))

    if not targets:
        targets = [("Пример", run_info[n//2][1])]

    fig, axes = plt.subplots(len(targets), 1, figsize=(14, 5 * len(targets)))
    if len(targets) == 1:
        axes = [axes]

    for ax, (label, (gap_s, gap_e)) in zip(axes, targets):
        actual_min = (gap_e - gap_s) * step / 60
        pad = int(min(240, max(60, actual_min * 3)) * 60 / step)
        w0  = max(0, gap_s - pad)
        w1  = min(len(ds), gap_e + pad)

        x_all = pd.to_datetime(ds[w0:w1])
        y_obs  = y_clean[w0:w1].astype(float)
        y_fill = y_filled[w0:w1].astype(float)
        mc     = method_codes[w0:w1]

        ax.plot(x_all, y_obs, color=C["obs"], lw=1.2,
                label="Исходные данные", zorder=3)

        fill_mask = mc > 0
        if fill_mask.any():
            ax.plot(x_all[fill_mask], y_fill[fill_mask],
                    color=C["nhits"], lw=2.0, ls="--",
                    label="Заполнение (NHITS)", zorder=5)

        xl = pd.to_datetime(ds[gap_s])
        xr = pd.to_datetime(ds[min(gap_e, len(ds)-1)])
        ax.axvspan(xl, xr, color=C["gap_bg"], alpha=0.6, zorder=1,
                   label=f"Зона пропуска ({actual_min:.0f} мин)")
        ax.axvline(xl, color=C["nhits"], lw=0.9, ls=":")
        ax.axvline(xr, color=C["nhits"], lw=0.9, ls=":")

        mae_v = np.nan
        gt = y_clean[gap_s:gap_e].astype(float)
        pv = y_filled[gap_s:gap_e].astype(float)
        ok = ~np.isnan(gt) & ~np.isnan(pv)
        if ok.sum() > 1:
            mae_v = np.nanmean(np.abs(gt[ok] - pv[ok]))
            ax.text(0.99, 0.04, f"MAE = {mae_v:,.0f} нТл",
                    transform=ax.transAxes, ha="right", va="bottom",
                    fontsize=9, color=C["nhits"],
                    bbox=dict(boxstyle="round,pad=0.3", fc="white", alpha=0.85))

        ts = pd.to_datetime(ds[gap_s]).strftime("%Y-%m-%d %H:%M")
        ax.set_title(f"{label} — {actual_min:.0f} мин ({ts})", fontweight="bold")
        ax.set_ylabel("Магнитное поле, нТл", fontsize=10)
        ax.legend(loc="upper left", ncol=3, fontsize=8)
        _fmt_xaxis(ax, x_all)

    axes[-1].set_xlabel("Время", fontsize=10)
    fig.suptitle("Примеры заполнения пропусков", fontsize=13, fontweight="bold", y=1.01)
    fig.tight_layout()
    out = os.path.join(out_dir, "fig_examples.png")
    fig.savefig(out, dpi=150, bbox_inches="tight")
    plt.close(fig)
    print(f"Saved: {out}")


# ── График 3: метрики ────────────────────────────────────────────────────────
def fig_metrics(metrics_df, out_dir=OUT_DIR):
    order = ["<15m", "15-60m", "1-3h", ">3h"]
    df = metrics_df.copy()
    present = [b for b in order if b in df["bucket"].values]
    if not present:
        print("[fig_metrics] No data, skipping.")
        return
    df = df[df["bucket"].isin(present)].copy()
    df["bucket"] = pd.Categorical(df["bucket"], categories=present, ordered=True)
    df = df.sort_values("bucket").reset_index(drop=True)

    x = np.arange(len(df))
    w = 0.3
    fig, ax = plt.subplots(figsize=(10, 6))
    b1 = ax.bar(x - w/2, df["MAE"],  w, label="MAE",  color=C["obs"],   alpha=0.85, edgecolor="white")
    b2 = ax.bar(x + w/2, df["RMSE"], w, label="RMSE", color=C["nhits"], alpha=0.85, edgecolor="white")
    for bar in list(b1) + list(b2):
        h = bar.get_height()
        if not np.isnan(h):
            ax.text(bar.get_x() + bar.get_width()/2, h + max(df["RMSE"].max(), 1) * 0.01,
                    f"{h:,.0f}", ha="center", va="bottom", fontsize=8)
    ax.set_xticks(x)
    ax.set_xticklabels(df["bucket"].tolist(), fontsize=10)
    ax.set_xlabel("Длительность пропуска")
    ax.set_ylabel("Ошибка, нТл")
    ax.set_title("Метрики NHITS по категориям пропусков", fontweight="bold", fontsize=12)
    ax.legend()
    fig.tight_layout()
    out = os.path.join(out_dir, "fig_metrics.png")
    fig.savefig(out, dpi=150, bbox_inches="tight")
    plt.close(fig)
    print(f"Saved: {out}")


# ── График 4: overview ───────────────────────────────────────────────────────
def fig_overview(ds, y_clean, y_filled, method_codes, cfg, out_dir=OUT_DIR):
    step = cfg.step_seconds

    lo = np.nanpercentile(y_clean, 0.5)
    hi = np.nanpercentile(y_clean, 99.5)

    thin  = max(1, len(ds) // 8000)
    x_all = pd.to_datetime(ds[::thin])
    y_obs = np.clip(y_clean[::thin].astype(float), lo, hi)
    yf    = np.clip(y_filled[::thin].astype(float), lo, hi)
    mc    = (method_codes[::thin] > 0)

    fig = plt.figure(figsize=(16, 8))
    gs  = gridspec.GridSpec(2, 2, figure=fig, height_ratios=[3, 1.5],
                            hspace=0.45, wspace=0.3)
    ax_main = fig.add_subplot(gs[0, :])
    ax_pct  = fig.add_subplot(gs[1, 0])
    ax_cum  = fig.add_subplot(gs[1, 1])

    ax_main.plot(x_all, y_obs, color=C["obs"], lw=0.5, alpha=0.85,
                 label="Исходные данные", zorder=2)
    if mc.any():
        ax_main.scatter(x_all[mc], yf[mc], s=5, color=C["nhits"],
                        alpha=0.8, label="Заполненные точки", zorder=4)
    else:
        ax_main.text(0.5, 0.5, "Нет заполненных точек",
                     transform=ax_main.transAxes, ha="center", color="gray")

    ax_main.set_ylabel("Магнитное поле, нТл", fontsize=11)
    ax_main.set_title("Временной ряд 2024: исходные данные и заполнение",
                      fontweight="bold", fontsize=13)
    ax_main.legend(ncol=2, fontsize=9)
    _fmt_xaxis(ax_main, x_all)

    df_m = pd.DataFrame({
        "ds":     pd.to_datetime(ds),
        "filled": (method_codes > 0).astype(int)
    })
    df_m["month"] = df_m["ds"].dt.to_period("M")
    pct = df_m.groupby("month")["filled"].mean() * 100
    bars = ax_pct.bar(range(len(pct)), pct.values,
                      color=C["nhits"], alpha=0.8, edgecolor="white")
    ax_pct.set_xticks(range(len(pct)))
    ax_pct.set_xticklabels([str(m)[-2:] for m in pct.index], fontsize=8)
    ax_pct.set_ylabel("% заполнено")
    ax_pct.set_xlabel("Месяц 2024")
    ax_pct.set_title("Доля заполненных точек по месяцам", fontweight="bold")
    for bar, v in zip(bars, pct.values):
        if v > 0.2:
            ax_pct.text(bar.get_x() + bar.get_width()/2,
                        bar.get_height() + 0.02,
                        f"{v:.1f}%", ha="center", va="bottom", fontsize=7)

    cum = df_m.groupby("month")["filled"].sum().cumsum()
    ax_cum.plot(range(len(cum)), cum.values, color=C["nhits"], lw=2, marker="o", ms=5)
    ax_cum.fill_between(range(len(cum)), cum.values, alpha=0.15, color=C["nhits"])
    ax_cum.set_xticks(range(len(cum)))
    ax_cum.set_xticklabels([str(m)[-2:] for m in cum.index], fontsize=8)
    ax_cum.set_ylabel("Точек (накопл.)")
    ax_cum.set_xlabel("Месяц 2024")
    ax_cum.set_title("Накопленное число заполненных точек", fontweight="bold")
    total = int(cum.values[-1]) if len(cum) else 0
    ax_cum.yaxis.set_major_formatter(
        mticker.FuncFormatter(lambda x, _: f"{x/1e3:.1f}k" if x >= 1000 else f"{int(x)}"))
    ax_cum.text(0.97, 0.05, f"Итого: {total:,}",
                transform=ax_cum.transAxes, ha="right", va="bottom",
                fontsize=9, color=C["nhits"])

    out = os.path.join(out_dir, "fig_overview.png")
    fig.savefig(out, dpi=150, bbox_inches="tight")
    plt.close(fig)
    print(f"Saved: {out}")


def save_all_plots(ds, y_clean, y_filled, gap_runs, method_codes, metrics_df, cfg,
                   nf_calm=None, nf_storm=None, hourly_medians=None, is_storm=None,
                   out_dir=OUT_DIR):
    print("Saving figures...")
    for gap_h in [0.5, 1.0, 3.0]:
        fig_comparison(ds, y_clean, y_filled, method_codes,
                       nf_calm, nf_storm, hourly_medians, is_storm,
                       cfg, gap_hours=gap_h, out_dir=out_dir)
    fig_examples(ds, y_clean, y_filled, method_codes, cfg, out_dir)
    fig_metrics(metrics_df, out_dir)
    fig_overview(ds, y_clean, y_filled, method_codes, cfg, out_dir)
    print(f"All figures saved to {out_dir}/")


## Запуск

In [ ]:
def main() -> None:
    args = parse_args()
    year = int(RUN_YEAR)
    csv_path = CSV_PATH
    out_dir = OUT_DIR
    device_clear_cuda = bool(DEVICE_CLEAR_CUDA)

    cfg = ImputeConfig()

    if not os.path.exists(csv_path):
        raise FileNotFoundError(f"Не найден файл: {csv_path}")

    os.makedirs(out_dir, exist_ok=True)

    if cfg.step_seconds != 3:
        print(f"[WARN] step_seconds={cfg.step_seconds} (в идеале 3 по вашему датасету).")

    # Длина чанка — именно на нее обучается NHITS
    h_train_steps = int((cfg.nhits_chunk_minutes * 60) // cfg.step_seconds)
    print(f"Train horizon chunk: {cfg.nhits_chunk_minutes} min => h={h_train_steps} steps")
    print(
        f"[CFG] input_size={cfg.input_size}, nhits_chunk_minutes={cfg.nhits_chunk_minutes} => h={h_train_steps}, "
        f"max_steps={cfg.max_steps}, batch_size={cfg.batch_size}, windows_batch_size={cfg.windows_batch_size}, "
        f"long_gap_mode={cfg.long_gap_mode}"
    )

    if device_clear_cuda and torch.cuda.is_available():
        torch.cuda.empty_cache()

    print("Loading series...")
    seconds_full, ds, y_raw, is_bad = load_ars_series(csv_path, year, cfg)
    y_clean = y_raw.copy()
    y_clean[is_bad] = np.nan

    print(f"Total points: {len(ds):,}")
    print(f"Missing/bad points: {np.isnan(y_clean).sum():,} ({np.isnan(y_clean).mean()*100:.2f}%)")

    print("Classifying storm/calm (rolling std 24h)...")
    is_storm = compute_is_storm(y_clean, cfg, step_seconds=cfg.step_seconds)
    print(f"Storm share: {is_storm.mean()*100:.2f}%")

    # Медианы по часу суток для фолбэка (считаем по "хорошим" точкам)
    print("Computing hourly medians (month, hour) for fallback...")
    hourly_medians = compute_hourly_medians(ds, y_raw, is_bad)

    print("Подготовка сегментов для обучения...")
    segs = split_into_segments(
        y_clean=y_clean,
        is_storm=is_storm,
        ds=ds,
        input_size=cfg.input_size,
        h_train=h_train_steps,
        max_series=cfg.max_series_per_label,
        max_points_per_series=cfg.max_points_per_series,
    )
    calm_df = segs["calm"]
    storm_df = segs["storm"]
    print(f"calm training rows: {len(calm_df):,} | storm training rows: {len(storm_df):,}")
    if len(calm_df) < cfg.input_size + h_train_steps:
        raise RuntimeError("Недостаточно данных для обучения calm модели. Уменьшите фильтры/лимиты.")
    if len(storm_df) < cfg.input_size + h_train_steps:
        raise RuntimeError("Недостаточно данных для обучения storm модели. Уменьшите фильтры/лимиты.")

    print("Обучение NHITS для calm...")
    nf_calm = train_nhits(calm_df, h_train_steps, cfg)
    gc.collect()
    if device_clear_cuda and torch.cuda.is_available():
        torch.cuda.empty_cache()

    print("Обучение NHITS для storm...")
    nf_storm = train_nhits(storm_df, h_train_steps, cfg)
    gc.collect()
    if device_clear_cuda and torch.cuda.is_available():
        torch.cuda.empty_cache()

    print("Imputing actual gaps...")
    y_filled = y_clean.copy()
    method_codes = np.zeros_like(y_filled, dtype=np.uint8)
    # На хороших точках code остается 0 (они не будут замаскированы)
    missing_mask = np.isnan(y_filled)
    print(f"y_clean NaN: {np.isnan(y_clean).sum():,}")
    print(f"y_filled NaN: {np.isnan(y_filled).sum():,}")
    print(f"missing_mask sum: {missing_mask.sum():,}")
    gap_runs = find_nan_runs(missing_mask)
    print(f"Found {len(gap_runs)} missing gap runs.")

    # Идем по пропускам последовательно, чтобы контекст постепенно обновлялся
    for i, (s, e) in enumerate(gap_runs):
        if e - s <= 0:
            continue
        impute_gap(
            y_inout=y_filled,
            ds=ds,
            is_storm=is_storm,
            nf_calm=nf_calm,
            nf_storm=nf_storm,
            hourly_medians=hourly_medians,
            gap_start=s,
            gap_end=e,
            cfg=cfg,
            method_codes=method_codes,
        )
        if (i + 1) % 10 == 0:
            print(f"  done {i+1}/{len(gap_runs)} gaps...")

    # Сохраняем заполненный ряд
    filled_df = pd.DataFrame(
        {
            "seconds": seconds_full,
            "ds": pd.to_datetime(ds),
            "value_raw": y_raw,
            "value_clean": y_clean,
            "value_filled": y_filled,
            "method_code": method_codes,
            "is_storm": is_storm.astype(np.uint8),
        }
    )
    out_filled_csv = os.path.join(out_dir, f"ARS_pos1_{year}_imputed_nhits.csv")
    filled_df.to_csv(out_filled_csv, index=False)
    print(f"Saved: {out_filled_csv}")

    # Метрики (оценка через синтетическое маскирование)
    print("Evaluating imputation quality with synthetic masks...")
    metrics_df = eval_imputation(
        y_clean=y_clean,
        ds=ds,
        is_storm=is_storm,
        nf_calm=nf_calm,
        nf_storm=nf_storm,
        hourly_medians=hourly_medians,
        cfg=cfg,
    )
    out_metrics_csv = os.path.join(out_dir, f"metrics_synthetic_{year}.csv")
    metrics_df.to_csv(out_metrics_csv, index=False)
    print(f"Saved: {out_metrics_csv}")

    save_all_plots(
        ds=ds,
        y_clean=y_clean,
        y_filled=y_filled,
        gap_runs=gap_runs,
        method_codes=method_codes,
        metrics_df=metrics_df,
        cfg=cfg,
        nf_calm=nf_calm,
        nf_storm=nf_storm,
        hourly_medians=hourly_medians,
        is_storm=is_storm,
        out_dir=out_dir,
    )

    print("\nDone.")
    print(method_codes.max(), method_codes.min())
    print(np.isnan(y_clean).sum(), np.isnan(y_filled).sum())


if __name__ == "__main__":
    main()